# Bibliothèque Scientifique — Nexus Science
## Scanner Open Library V2
Version revue : comptage des lignes physiques, checkpoint atomique, reprise par troncature à la dernière position validée, contrôle de l'identité de la source et filtre chronologique 1720–2026.

**Précaution :** ce notebook ne se connecte pas à Neon et n'effectue aucun transfert de données vers une base. Il écrit uniquement dans le stockage de travail Kaggle. La reprise relit le flux gzip depuis le début jusqu'au checkpoint.

In [ ]:
import os, sys, platform
print('Python :', sys.version)
print('Plateforme :', platform.platform())
print('Dossier Kaggle :', os.path.isdir('/kaggle/working'))

### Scanner complet
Exécuter cette cellule pour lancer ou reprendre le traitement. Pour interrompre, arrêter simplement l'exécution : le dernier checkpoint validé permettra de reprendre sans conserver les écritures postérieures à ce checkpoint.

In [ ]:
# Nexus Science — Open Library Editions scanner V2
# Streaming gzip + checkpoint atomique + reprise sans doublons.
# Ce script écrit uniquement dans /kaggle/working ; aucune connexion à Neon.

import os
import re
import json
import gzip
import time
import urllib.request
from pathlib import Path

BASE = Path("/kaggle/working/nexus_science")
BASE.mkdir(parents=True, exist_ok=True)

URL = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
OUT = BASE / "scientific_editions_v2.jsonl"
CHECKPOINT = BASE / "scanner_checkpoint_v2.json"
BATCH_LINES = 10_000
MIN_YEAR = 1720
MAX_YEAR = 2026
USER_AGENT = "NexusScience-Research/2.0"

# Les mots-clés sont appliqués au titre et aux sujets, sans supposer
# que le nom de la langue d'une notice soit toujours disponible.
KEYWORDS = (
    "science", "scientific", "scientifique", "sciences",
    "mathematics", "mathematical", "mathematique", "mathématiques",
    "physics", "physique", "chemistry", "chimie",
    "biology", "biologie", "biological", "medicine", "medical", "médecine",
    "health science", "anatomy", "anatomie", "engineering", "ingénierie",
    "computer science", "informatique", "astronomy", "astronomie",
    "astrophysics", "astrophysique", "geology", "géologie",
    "geography", "géographie", "ecology", "écologie",
    "environmental science", "botany", "botanique", "zoology", "zoologie",
    "statistics", "statistique", "microbiology", "microbiologie",
    "genetics", "génétique", "pharmacology", "pharmacologie",
    "neuroscience", "neurosciences", "agriculture", "agronomie",
    "veterinary", "vétérinaire", "biotechnology", "biotechnologie",
    "ciencia", "ciencias", "matematica", "matemática", "fisica",
    "física", "quimica", "química", "biologia", "biología",
    "medicina", "ingenieria", "engenharia", "ciência", "astronomia",
)

# Exclusions limitées au titre pour éviter de supprimer trop de notices.
NOISE_TITLE = (
    "statutory instrument", "parliamentary debates",
    "parliamentary proceedings", "government notice",
    "house of lords bill",
)
YEAR_RE = re.compile(r"(?<!\d)(1[7-9]\d{2}|20\d{2})(?!\d)")

def source_identity(response):
    """Métadonnées permettant de détecter un dump Open Library remplacé."""
    return {
        "url": URL,
        "etag": response.headers.get("ETag"),
        "last_modified": response.headers.get("Last-Modified"),
        "content_length": response.headers.get("Content-Length"),
    }

def atomic_checkpoint(state):
    """Écrit le checkpoint temporairement, synchronise puis remplace l'ancien."""
    tmp = CHECKPOINT.with_suffix(CHECKPOINT.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, CHECKPOINT)
    # Synchroniser le répertoire quand le système le permet.
    try:
        fd = os.open(str(CHECKPOINT.parent), os.O_DIRECTORY)
        try:
            os.fsync(fd)
        finally:
            os.close(fd)
    except (AttributeError, OSError):
        pass

def get_year(record):
    date_text = str(record.get("publish_date") or "")
    match = YEAR_RE.search(date_text)
    return int(match.group(1)) if match else None

def is_scientific(record):
    title = str(record.get("title") or "").casefold()
    subjects = record.get("subjects") or []
    if not isinstance(subjects, list):
        subjects = []
    subject_text = " ".join(s for s in subjects if isinstance(s, str)).casefold()
    searchable = title + " " + subject_text
    if any(term in title for term in NOISE_TITLE):
        return False
    return any(term.casefold() in searchable for term in KEYWORDS)

def load_state(identity):
    if not CHECKPOINT.exists():
        if OUT.exists() and OUT.stat().st_size:
            raise RuntimeError(
                "Un JSONL existe sans checkpoint V2. Sauvegardez-le et "
                "ne le mélangez pas à une nouvelle exécution."
            )
        return {
            "version": 2,
            "source": identity,
            "scan_lines": 0,
            "valid_json": 0,
            "selected": 0,
            "invalid_json": 0,
            "out_of_period": 0,
            "output_bytes": 0,
        }

    with open(CHECKPOINT, "r", encoding="utf-8") as f:
        state = json.load(f)
    if state.get("version") != 2:
        raise RuntimeError("Version de checkpoint incompatible.")
    if state.get("source") != identity:
        raise RuntimeError(
            "La signature du dump a changé. Ne reprenez pas ce checkpoint "
            "avec une autre version de la source."
        )
    if not OUT.exists():
        raise RuntimeError("Checkpoint présent mais fichier JSONL absent.")
    actual_size = OUT.stat().st_size
    expected_size = int(state["output_bytes"])
    if actual_size < expected_size:
        raise RuntimeError("Le JSONL est plus court que le checkpoint : reprise refusée.")
    # Annule toute écriture qui aurait eu lieu après le dernier checkpoint.
    with open(OUT, "r+b") as f:
        f.truncate(expected_size)
        f.flush()
        os.fsync(f.fileno())
    return state

def save_progress(state):
    state["output_bytes"] = OUT.stat().st_size if OUT.exists() else 0
    atomic_checkpoint(state)

def run_scanner():
    request = urllib.request.Request(
        URL, headers={"User-Agent": USER_AGENT}
    )
    started = time.time()

    with urllib.request.urlopen(request, timeout=180) as response:
        identity = source_identity(response)
        state = load_state(identity)
        resume_at = int(state["scan_lines"])

        print("NEXUS SCIENCE — SCANNER V2")
        print("Source :", URL)
        print("Lignes physiques déjà validées :", resume_at)
        print("Notices retenues cumulées :", state["selected"])
        print("Période :", MIN_YEAR, "à", MAX_YEAR)
        print("Sortie :", OUT)
        print("Reprise : relecture du flux depuis le début, puis saut des lignes checkpointées.")

        # Sauvegarde initiale : garantit un point de reprise même au tout début.
        save_progress(state)

        with open(OUT, "ab") as output:
            with gzip.GzipFile(fileobj=response, mode="rb") as gz:
                for physical_line, raw_line in enumerate(gz, start=1):
                    if physical_line <= resume_at:
                        if physical_line % 1_000_000 == 0:
                            print("Relecture de reprise :", physical_line)
                        continue

                    state["scan_lines"] = physical_line
                    parts = raw_line.rstrip(b"\r\n").split(b"\t")
                    if len(parts) < 4:
                        state["invalid_json"] += 1
                    else:
                        try:
                            record = json.loads(parts[-1])
                            if not isinstance(record, dict):
                                raise ValueError("La notice JSON n'est pas un objet")
                            state["valid_json"] += 1
                            year = get_year(record)
                            if year is None or not (MIN_YEAR <= year <= MAX_YEAR):
                                state["out_of_period"] += 1
                            elif is_scientific(record):
                                payload = (
                                    json.dumps(record, ensure_ascii=False, separators=(",", ":"))
                                    + "\n"
                                ).encode("utf-8")
                                output.write(payload)
                                state["selected"] += 1
                        except (json.JSONDecodeError, UnicodeDecodeError, ValueError, TypeError):
                            state["invalid_json"] += 1

                    if physical_line % BATCH_LINES == 0:
                        # Ordre essentiel : données durables d'abord, checkpoint ensuite.
                        output.flush()
                        os.fsync(output.fileno())
                        save_progress(state)
                        elapsed = round(time.time() - started, 1)
                        print(
                            f"Lignes : {state['scan_lines']:,} | "
                            f"JSON valides : {state['valid_json']:,} | "
                            f"retenues cumulées : {state['selected']:,} | "
                            f"hors période/sans date : {state['out_of_period']:,} | "
                            f"JSON invalides : {state['invalid_json']:,} | "
                            f"secondes : {elapsed}"
                        )

            output.flush()
            os.fsync(output.fileno())

    save_progress(state)
    print("\nSCAN TERMINÉ")
    print("Lignes physiques :", f"{state['scan_lines']:,}")
    print("JSON valides :", f"{state['valid_json']:,}")
    print("Notices scientifiques retenues :", f"{state['selected']:,}")
    print("Hors période ou sans année :", f"{state['out_of_period']:,}")
    print("JSON invalides / lignes mal formées :", f"{state['invalid_json']:,}")
    print("Taille JSONL (GiB) :", round(OUT.stat().st_size / 1024**3, 3))
    print("Checkpoint :", CHECKPOINT)
    print("Durée de cette session (s) :", round(time.time() - started, 1))


def self_test():
    """Tests locaux rapides : filtre de dates, sélection et reprise du checkpoint."""
    import tempfile

    assert get_year({"publish_date": "circa 1895"}) == 1895
    assert get_year({"publish_date": "2027"}) == 2027
    assert get_year({"publish_date": "date inconnue"}) is None
    assert is_scientific({"title": "Introduction to Physics"})
    assert not is_scientific({"title": "Ordinary literature"})

    global OUT, CHECKPOINT
    old_out, old_checkpoint = OUT, CHECKPOINT
    try:
        with tempfile.TemporaryDirectory(prefix="nexus_v2_test_") as tmp:
            OUT = Path(tmp) / "test.jsonl"
            CHECKPOINT = Path(tmp) / "test_checkpoint.json"
            identity = {
                "url": "https://example.invalid/test.gz",
                "etag": "test-etag",
                "last_modified": "test-date",
                "content_length": "123",
            }
            committed = b'{"title":"Physics"}\\n'
            OUT.write_bytes(committed + b'UNCOMMITTED_TAIL')
            state = {
                "version": 2,
                "source": identity,
                "scan_lines": 10,
                "valid_json": 9,
                "selected": 1,
                "invalid_json": 1,
                "out_of_period": 2,
                "output_bytes": len(committed),
            }
            atomic_checkpoint(state)
            recovered = load_state(identity)
            assert recovered["scan_lines"] == 10
            assert OUT.read_bytes() == committed, "La queue non validée n'a pas été tronquée"

            try:
                load_state({**identity, "etag": "different-etag"})
            except RuntimeError:
                pass
            else:
                raise AssertionError("Une source différente aurait dû être refusée")

            OUT.write_bytes(b"short")
            try:
                load_state(identity)
            except RuntimeError:
                pass
            else:
                raise AssertionError("Un JSONL plus court que le checkpoint aurait dû être refusé")
    finally:
        OUT, CHECKPOINT = old_out, old_checkpoint

    print("AUTO-TEST V2 : OK — dates, mots-clés, troncature, identité source et fichier trop court.")

self_test()

if __name__ == "__main__":
    run_scanner()


### Auto-test avant le scan\nLa cellule du scanner lance d'abord un test local rapide (sans réseau et sans toucher aux fichiers de production). Il vérifie le filtre de dates/mots-clés, la troncature des écritures non checkpointées, le refus d'une source différente et le refus d'un JSONL trop court. Le test ne remplace pas un essai réel sur un petit échantillon Open Library.\n

### Vérification des fichiers produits
Cette cellule contrôle la présence du JSONL et du checkpoint, ainsi que la cohérence de la taille enregistrée.

In [ ]:
import json
from pathlib import Path
base = Path('/kaggle/working/nexus_science')
out = base / 'scientific_editions_v2.jsonl'
cp = base / 'scanner_checkpoint_v2.json'
print('JSONL présent :', out.exists())
print('Checkpoint présent :', cp.exists())
if out.exists(): print('Taille JSONL (GiB) :', round(out.stat().st_size / 1024**3, 3))
if cp.exists():
    state = json.loads(cp.read_text(encoding='utf-8'))
    print('Lignes checkpointées :', state['scan_lines'])
    print('Notices retenues :', state['selected'])
    print('Octets de sortie checkpointés :', state['output_bytes'])
    if out.exists(): print('Taille cohérente :', out.stat().st_size >= state['output_bytes'])